## Question 2: Metropolis Algorithm

Given $S(x) = x^4$, the target distribution is:

$$f(x) = \frac{1}{Z} e^{-x^4}, \quad x \in \mathbb{R}$$

### 2a. Estimate Normalization Factor Z

The normalization constant $Z$ ensures $f(x)$ integrates to 1:

$$Z = \int_{-\infty}^{\infty} e^{-x^4} \, dx$$

Since $e^{-x^4}$ is even, $Z = 2 \int_0^{\infty} e^{-x^4} \, dx$.

Let $u = x^4$, so $x = u^{1/4}$ and $dx = \frac{1}{4} u^{-3/4} \, du$:

$$Z = \frac{2}{4} \int_0^{\infty} u^{-3/4} e^{-u} \, du = \frac{1}{2} \int_0^{\infty} u^{1/4 - 1} e^{-u} \, du = \frac{1}{2} \, \Gamma\!\left(\frac{1}{4}\right)$$

Since $f(x) \propto e^{-x^4}$ is symmetric about $x = 0$, we have $E[X] = 0$.

In [ ]:
import matplotlib
matplotlib.use('Agg')
import os
os.makedirs('2', exist_ok=True)
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.special import gamma
from scipy.optimize import minimize_scalar

np.random.seed(42)

def target_f(x):
    return np.exp(-x**4)

Z_exact = 0.5 * gamma(0.25)
theoretical_var = gamma(0.75) / gamma(0.25)
sigma_target = np.sqrt(theoretical_var)

print(f"Theoretical Normalization Constant Z: {Z_exact:.4f}")
print(f"Theoretical Expectation (Mean): 0.0000")
print(f"Theoretical Variance: {theoretical_var:.4f}")
print(f"sigma_target = sqrt(Var(X)) = {sigma_target:.4f}")
print("-" * 30)

### 2b. Metropolis Sampling with Three Symmetric Proposals

Roberts, Gelman & Gilks (1997) established the optimal scaling framework for random-walk Metropolis algorithms, showing that the proposal variance can be chosen to maximize convergence speed. Gelman, Roberts & Gilks (1996, Table 1, p. 605) report the numerically optimal acceptance rate for dimension $d = 1$ as approximately 44%, achieved with proposal scale:

$$\sigma_{\text{opt}} = 2.38 \cdot \sigma_{\text{target}}, \quad \sigma_{\text{target}} = \sqrt{\text{Var}(X)}$$

Since we are sampling a 1D target $f(x) \propto e^{-x^4}$, we use this result. We apply $\sigma_{\text{opt}}$ uniformly to all three symmetric proposal distributions: Gaussian, Uniform, and Laplace.

**References:**
- Roberts, G.O., Gelman, A. & Gilks, W.R. (1997). Weak convergence and optimal scaling of random walk Metropolis algorithms. *Annals of Applied Probability*, 7(1), 110–120.
- Gelman, A., Roberts, G.O. & Gilks, W.R. (1996). Efficient Metropolis jumping rules. In *Bayesian Statistics 5*, 599–607. Oxford University Press.

In [ ]:
N_samples = 1000

sigma_opt = 2.38 * sigma_target
print(f"sigma_opt = 2.38 * {sigma_target:.4f} = {sigma_opt:.4f}")

x_grid = np.linspace(-2.5, 2.5, 500)
true_density = target_f(x_grid) / Z_exact

proposal_configs = [
    ('Gaussian', 'gaussian'),
    ('Uniform', 'uniform'),
    ('Laplace', 'laplace')
]

def metropolis_sampler_all(N, step_size, proposal_type, x0=0.0, seed=42):
    rng = np.random.default_rng(seed)
    samples = np.zeros(N)
    samples[0] = x0
    accepted = 0

    for t in range(1, N):
        x_curr = samples[t-1]

        # Step 1: Propose y from symmetric q(y|x)
        if proposal_type == 'gaussian':
            x_prop = rng.normal(loc=x_curr, scale=step_size)
        elif proposal_type == 'uniform':
            x_prop = rng.uniform(low=x_curr - step_size, high=x_curr + step_size)
        elif proposal_type == 'laplace':
            x_prop = rng.laplace(loc=x_curr, scale=step_size)
        else:
            raise ValueError("proposal_type must be 'gaussian', 'uniform', or 'laplace'")

        # Step 2: Acceptance ratio alpha = min(1, pi(y)/pi(x))
        # Symmetric proposal: Hastings correction = 1
        # Unnormalized density pi(x) = e^{-x^4}, Z cancels in ratio
        ratio = target_f(x_prop) / target_f(x_curr)
        alpha = min(1.0, ratio)

        # Step 3: Accept or reject
        if rng.uniform(0, 1) <= alpha:
            samples[t] = x_prop
            accepted += 1
        else:
            samples[t] = x_curr

    acceptance_rate = accepted / (N - 1)
    return samples, acceptance_rate

def compute_metrics(samples, acc_rate, step_size):
    return {
        'Step Size': round(step_size, 4),
        'Acceptance Rate (%)': f"{acc_rate * 100:.2f}%",
        'Sample Mean': round(np.mean(samples), 4),
        'Sample Variance': round(np.var(samples), 4)
    }

def plot_proposals(samples_dict, results_dict, title, save_path=None):
    fig, axs = plt.subplots(3, 2, figsize=(14, 12))
    colors = ['navy', 'forestgreen', 'darkorange']

    for i, (name, samples) in enumerate(samples_dict.items()):
        axs[i, 0].plot(samples, color=colors[i], alpha=0.8, linewidth=0.8)
        axs[i, 0].set_title(f"Trace Plot - {name} Proposal", fontsize=11)
        axs[i, 0].set_xlabel("Iteration Step", fontsize=9)
        axs[i, 0].set_ylabel("x value", fontsize=9)
        axs[i, 0].grid(True, alpha=0.3)

        axs[i, 1].hist(samples, bins=30, density=True, color='skyblue',
                        edgecolor='black', alpha=0.6, label='MCMC Samples')
        axs[i, 1].plot(x_grid, true_density, 'r-', lw=2, label=r'True Density $f(x)/Z$')
        axs[i, 1].set_title(f"Distribution - {name} Proposal", fontsize=11)
        axs[i, 1].set_xlabel("x", fontsize=9)
        axs[i, 1].set_ylabel("Density", fontsize=9)
        axs[i, 1].legend(fontsize=9)
        axs[i, 1].grid(True, alpha=0.3)

    plt.suptitle(f"{title} (N = {N_samples})", fontsize=14, fontweight='bold', y=1.02)
    plt.tight_layout()
    if save_path:
        plt.savefig(save_path, dpi=150, bbox_inches='tight')
        print(f"Saved {save_path}")
    plt.close()

In [ ]:
initial_samples = {}
initial_results = {}

for name, ptype in proposal_configs:
    samples, acc_rate = metropolis_sampler_all(
        N=N_samples,
        step_size=sigma_opt,
        proposal_type=ptype
    )
    initial_samples[name] = samples
    initial_results[name] = compute_metrics(samples, acc_rate, sigma_opt)

print(f"All proposals run with sigma_opt = {sigma_opt:.4f}")

**Performance Metrics**

| Metric | What It Measures | Optimal Value |
|---|---|---|
| Acceptance Rate | Fraction of proposals accepted | $\approx 44\%$ for $d=1$ (Roberts et al., 1997) |
| Sample Mean (E[X]) | First moment estimate | $0.0000$ (by symmetry of $f$) |
| Sample Variance | Second moment estimate | $\Gamma(3/4)/\Gamma(1/4)$ |

In [ ]:
print(f"Results with sigma_opt = {sigma_opt:.4f} for all proposals")
df1 = pd.DataFrame(initial_results).T
df1.loc['Theoretical'] = ['--', '~44.00%', 0.0, round(theoretical_var, 4)]
display(df1)
plot_proposals(initial_samples, initial_results,
               f"sigma_opt = {sigma_opt:.4f}", save_path="2/initial_proposals.png")

### 2c. Algorithm Explanation and Numerical Performance

The **Metropolis-Hastings algorithm** is a Markov Chain Monte Carlo (MCMC) method used for sampling from probability distributions where direct sampling is difficult or impossible. It works by constructing a Markov chain whose stationary distribution is the target distribution $\pi(x)$.

**The core steps of the Metropolis Algorithm are:**

1.  **Initialization:** Start with an initial state $x_0$.
2.  **Proposal Generation:** At each step $t$, generate a candidate sample $y$ from a proposal distribution $q(y | x_t)$, where $x_t$ is the current state.
3.  **Acceptance Ratio Calculation:** Calculate the acceptance ratio $\alpha$:
    $$\alpha = \min\left(1, \frac{\pi(y)}{\pi(x_t)}\right)$$
4.  **Acceptance:** Generate a random number $u \sim U(0, 1)$.
    *   If $u \le \alpha$, accept the candidate: $x_{t+1} = y$.
    *   Otherwise, reject the candidate: $x_{t+1} = x_t$.


**How it works with different symmetric proposal distributions:**

*   **Gaussian:** This proposal allows for smooth, continuous jumps from the current state. A larger `step_size` leads to bigger proposed jumps, which can explore the sample space faster but might result in lower acceptance rates if the jumps are too large and move to low-probability regions. A smaller `step_size` results in higher acceptance but slower exploration.

*   **Uniform Proposal:** This proposal generates candidates within a strict bounded range. Similar to Gaussian, `step_size` affects the exploration-acceptance trade-off. Large uniform steps can quickly jump to different parts of the space, but might struggle to fine-tune sampling in narrow, high-density regions.

*   **Laplace Proposal:** This proposal exhibits dual behavior due to its sharp peak at the center and heavier tails compared to the Gaussian. The sharp peak generates many localized, very small steps, yielding high acceptance in nearby regions. Concurrently, the heavy tails occasionally propose large, long-range jumps. For the target $f(x) = e^{-x^4}$, these extreme jumps frequently overshoot into near-zero probability zones and get rejected, which can lead to temporary stagnation and higher sample autocorrelation.

**Performance Analysis with $\sigma_{\text{opt}} \approx 1.38$**

The $2.38 \cdot \sigma_{\text{target}}$ formula was derived by Roberts et al. (1997) under the assumption of a **Gaussian proposal** targeting a **Gaussian distribution**. Our target $f(x) \propto e^{-x^4}$ is super-Gaussian — it has lighter tails than a Gaussian, meaning the density decays faster. This changes the optimal step size landscape:

- **Gaussian proposal**: The formula is closest to optimal here, since the proposal matches the derivation's assumptions. The acceptance rate is near 44%.
- **Uniform proposal**: Uniform proposals have bounded support, so all proposals within the window have similar probability. This leads to systematically higher acceptance rates for the same step size.
- **Laplace proposal**: The heavy tails of the Laplace distribution generate occasional large jumps that get rejected, while its sharp peak generates many small accepted steps. This creates a different acceptance rate profile.

The formula does not generalize to non-Gaussian proposals or non-Gaussian targets, motivating per-proposal tuning in Section 2d.

### 2d. Step Size Tuning and Final Results

To improve upon the theoretical step size, we use `scipy.optimize.minimize_scalar` to find the step size that minimizes $|\text{acceptance rate} - 0.44|$ for each proposal independently. This is a scalar black-box optimization over the step size parameter.

In [ ]:
# Tune step size for each proposal to target 44% acceptance
tuned_step_sizes = {}

for name, ptype in proposal_configs:
    def objective(step_size, ptype=ptype):
        _, acc_rate = metropolis_sampler_all(N=N_samples, step_size=step_size,
                                              proposal_type=ptype)
        return abs(acc_rate - 0.44)

    result = minimize_scalar(objective, bounds=(0.1, 5.0), method='bounded')
    tuned_step_sizes[name] = result.x
    print(f"{name}: tuned step size = {result.x:.4f}")

print("\n" + "=" * 50)
print("Running with tuned step sizes")
print("=" * 50)

tuned_samples = {}
tuned_results = {}

for name, ptype in proposal_configs:
    step = tuned_step_sizes[name]
    samples, acc_rate = metropolis_sampler_all(
        N=N_samples,
        step_size=step,
        proposal_type=ptype
    )
    tuned_samples[name] = samples
    tuned_results[name] = compute_metrics(samples, acc_rate, step)

print("\nTuned Results")
df2 = pd.DataFrame(tuned_results).T
df2.loc['Theoretical'] = ['--', '~44.00%', 0.0, round(theoretical_var, 4)]
display(df2)

plot_proposals(tuned_samples, tuned_results, "Tuned Step Sizes", save_path="2/tuned_proposals.png")

#### Comparison: Before vs After Tuning

After tuning, each proposal achieves an acceptance rate closer to the theoretical optimum of 44%. Key observations:

**1. Acceptance Rate**: Tuning brings all proposals closer to the 44% target, confirming that the $2.38 \cdot \sigma$ formula is only optimal for Gaussian proposals on Gaussian targets.

**2. Sample Mean**: All proposals produce sample means near 0, consistent with the symmetry of $f(x) \propto e^{-x^4}$.

**3. Sample Variance**: The tuned step sizes produce sample variances closer to the theoretical value of $\Gamma(3/4)/\Gamma(1/4) \approx 0.3380$, indicating better exploration of the target distribution.

**4. Practical Takeaway**: The theoretical step size formula provides a reasonable starting point, but per-proposal optimization is necessary for best performance. Among the three proposals, the Uniform proposal tends to perform well due to its even exploration within a bounded window, while the Laplace proposal's heavy tails can cause occasional stagnation.